# Exoplanet Discovery Methods Analysis

**Research Question**: How well can machine learning classify exoplanet discovery methods from planetary and stellar properties?

**Task**: Classification (predict discovery method: Transit, Radial Velocity, Imaging, Microlensing, etc.)

**Innovation**: Compare 4 models using 3 feature sets from diverse exoplanet properties

## Phase 1: Setup & Initialization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

RESULTS_PATH = Path('../results')
RESULTS_PATH.mkdir(exist_ok=True)

print('Configuration complete')

In [ ]:
hypotheses = {
    'H1': 'Discovery method can be predicted from planetary and stellar characteristics',
    'H2': 'Tree-based models outperform KNN for this classification task',
    'H3': 'Combined orbital and stellar features give best performance',
    'H4': 'Transit method is most predictable (largest class with 74%)'
}

print('HYPOTHESES (Before Analysis)')
for key, hyp in hypotheses.items():
    print(f'{key}: {hyp}')

## Phase 2-3: Data Loading & Exploration

In [ ]:
# Load from GitHub URL (exoplanets_pscomppars dataset, 2026 update)
df = pd.read_csv('https://raw.githubusercontent.com/duochen/data-science-bootcamp/refs/heads/main/FinalProjectExamples/astronomy/exoplanet-discovery-methods/data/exoplanet_dataset/data/exoplanets_pscomppars_2026-09-12.csv')

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["disc_year"].min()}-{df["disc_year"].max()}')
print(f'\nTarget variable (discoverymethod):')
print(df['discoverymethod'].value_counts())
print(f'\nNumeric columns: {df.select_dtypes(include=[np.number]).shape[1]}')
print(f'Missing values: {df.isnull().sum().sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
top_methods = df['discoverymethod'].value_counts().head(6)
top_methods.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_title('Distribution of Exoplanet Discovery Methods (Top 6)', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Discovery Method')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(RESULTS_PATH / '01_discovery_methods.png', dpi=300, bbox_inches='tight')
plt.show()
print('[PASS] Visualization 1 saved')

## Phase 4-5: Data Preparation & Feature Sets

In [ ]:
# Get numeric columns (exclude non-predictive columns like name, hostname, facility)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Define feature sets:
# A: Orbital properties only
# B: Stellar properties only
# C: Combined (all numeric features)

feature_set_a = ['pl_orbper', 'pl_orbperlim', 'pl_eqt', 'pl_eqtlim']  # Orbital
feature_set_b = ['st_teff', 'st_rad', 'sy_dist']  # Stellar + distance
feature_set_c = ['disc_year', 'pl_orbper', 'pl_orbperlim', 'pl_rade', 'pl_radelim', 'pl_bmasse', 'pl_bmasselim', 'pl_eqt', 'pl_eqtlim', 'st_teff', 'st_rad', 'sy_dist']  # Combined

feature_sets = {
    'A (Orbital)': feature_set_a,
    'B (Stellar)': feature_set_b,
    'C (Combined)': feature_set_c
}

print('Feature sets defined:')
for name, feats in feature_sets.items():
    print(f'  {name}: {len(feats)} features')

# Filter to top 4 discovery methods (for balanced classification)
top_methods_list = df['discoverymethod'].value_counts().head(4).index.tolist()
df_filtered = df[df['discoverymethod'].isin(top_methods_list)].copy()

print(f'\nFiltered to top 4 methods: {len(df_filtered)} exoplanets')
print(df_filtered['discoverymethod'].value_counts())

In [ ]:
# Prepare data: drop rows with missing values, scale, split
df_clean = df_filtered[feature_set_c + ['discoverymethod']].dropna()

print(f'Clean dataset: {len(df_clean)} rows')
print(f'Features: {len(feature_set_c)}')
print(f'Classes: {df_clean["discoverymethod"].nunique()}')

# Prepare X and y
X = df_clean[feature_set_c]
y = df_clean['discoverymethod']

# Stratified split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTrain: {len(X_train)}, Test: {len(X_test)}')
print('Data prepared and scaled')

In [ ]:
# Prepare feature sets for all models
splits_by_feature_set = {}

for set_name, features in feature_sets.items():
    X_train_subset = X_train[features]
    X_test_subset = X_test[features]
    
    # Scale these specific features
    scaler_subset = StandardScaler()
    X_train_subset_scaled = scaler_subset.fit_transform(X_train_subset)
    X_test_subset_scaled = scaler_subset.transform(X_test_subset)
    
    splits_by_feature_set[set_name] = {
        'X_train': X_train_subset_scaled,
        'X_test': X_test_subset_scaled,
        'y_train': y_train.values,
        'y_test': y_test.values
    }

print(f'Feature set splits prepared')

## Phase 6-10: Model Training

In [ ]:
models = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=RANDOM_SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
}

results_all = {}

for set_name, split_data in splits_by_feature_set.items():
    results_all[set_name] = {}
    X_train_s = split_data['X_train']
    X_test_s = split_data['X_test']
    y_train_s = split_data['y_train']
    y_test_s = split_data['y_test']
    
    for model_name, model in models.items():
        model.fit(X_train_s, y_train_s)
        y_pred = model.predict(X_test_s)
        
        acc = accuracy_score(y_test_s, y_pred)
        f1 = f1_score(y_test_s, y_pred, average='weighted', zero_division=0)
        
        results_all[set_name][model_name] = {
            'accuracy': acc,
            'f1': f1,
            'model': model,
            'y_pred': y_pred
        }
        
        print(f'{set_name} - {model_name}: Accuracy={acc:.3f}, F1={f1:.3f}')

## Phase 11: Model Comparison

In [ ]:
# Create results dataframe
results_rows = []
for set_name, models_dict_split in results_all.items():
    for model_name, metrics in models_dict_split.items():
        results_rows.append({
            'FeatureSet': set_name,
            'Model': model_name,
            'Accuracy': metrics['accuracy'],
            'F1': metrics['f1']
        })

results_df = pd.DataFrame(results_rows)
print('\nModel Results (Best 12 configurations):')
print(results_df.nlargest(12, 'Accuracy')[['FeatureSet', 'Model', 'Accuracy', 'F1']].to_string(index=False))

results_df.to_csv(RESULTS_PATH / 'model_results.csv', index=False)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for set_name in feature_sets.keys():
    set_data = results_df[results_df['FeatureSet'] == set_name].sort_values('Model')
    ax1.plot(set_data['Model'], set_data['Accuracy'], marker='o', label=set_name, linewidth=2, markersize=8)

ax1.set_title('Accuracy by Model and Feature Set', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Model')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

for set_name in feature_sets.keys():
    set_data = results_df[results_df['FeatureSet'] == set_name].sort_values('Model')
    ax2.plot(set_data['Model'], set_data['F1'], marker='s', label=set_name, linewidth=2, markersize=8)

ax2.set_title('F1 Score by Model and Feature Set', fontsize=12, fontweight='bold')
ax2.set_ylabel('F1 Score')
ax2.set_xlabel('Model')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_PATH / '02_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('[PASS] Visualization 2 saved')

## Phase 12-13: Feature Importance & Error Analysis

In [ ]:
# Get best model
best_config = results_df.nlargest(1, 'Accuracy').iloc[0]
best_set = best_config['FeatureSet']
best_model_name = best_config['Model']

best_model = results_all[best_set][best_model_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
    feature_names = feature_sets[best_set]
    
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance
    }).sort_values('Importance', ascending=False)
    
    print(f'Top 10 Features ({best_set} - {best_model_name}):')
    print(importance_df.head(10).to_string(index=False))
    
    importance_df.to_csv(RESULTS_PATH / 'feature_importance.csv', index=False)
else:
    print(f'Linear model used - no feature importances')

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    fig, ax = plt.subplots(figsize=(10, 6))
    top_features = importance_df.head(10)
    ax.barh(range(len(top_features)), top_features['Importance'].values, 
            color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features['Feature'].values)
    ax.invert_yaxis()
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 10 Features - {best_set}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_PATH / '03_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('[PASS] Visualization 3 saved')

## Phase 14-16: Hypothesis Testing & Summary

In [ ]:
print('\nHYPOTHESIS TESTING')
print('='*60)

baseline_acc = results_df[results_df['Model'] == 'Baseline']['Accuracy'].mean()
best_acc = results_df['Accuracy'].max()
print(f'\nH1: Predictable from characteristics')
print(f'    Baseline: {baseline_acc:.3f}, Best: {best_acc:.3f}')
print(f'    Result: {"CONFIRMED" if best_acc > baseline_acc + 0.1 else "WEAK"}')

tree_results = results_df[results_df['Model'] == 'Decision Tree']['Accuracy'].mean()
knn_results = results_df[results_df['Model'] == 'KNN']['Accuracy'].mean()
print(f'\nH2: Tree > KNN')
print(f'    Tree: {tree_results:.3f}, KNN: {knn_results:.3f}')
print(f'    Result: {"CONFIRMED" if tree_results > knn_results else "NOT CONFIRMED"}')

feature_impact = results_df.groupby('FeatureSet')['Accuracy'].mean()
print(f'\nH3: Combined features best')
print(f'    Orbital: {feature_impact["A (Orbital)"]:.3f}, Stellar: {feature_impact["B (Stellar)"]:.3f}, Combined: {feature_impact["C (Combined)"]:.3f}')
print(f'    Result: {"CONFIRMED" if feature_impact["C (Combined)"] == feature_impact.max() else "NOT CONFIRMED"}')

print(f'\nH4: Transit method predictability')
transit_class_prop = (df_clean['discoverymethod'] == 'Transit').sum() / len(df_clean)
print(f'    Transit proportion: {transit_class_prop:.1%}')
print(f'    Result: {"CONFIRMED" if transit_class_prop > 0.50 else "NOT CONFIRMED"}')

In [ ]:
summary = f"""EXOPLANET DISCOVERY METHODS ANALYSIS - SUMMARY
{'='*60}

DATASET
-------
Records: {len(df_clean)} exoplanets
Features: {len(feature_set_c)} numeric attributes
Classes: {df_clean['discoverymethod'].nunique()} (top 4 methods)
Date range: {df_filtered['disc_year'].min()}-{df_filtered['disc_year'].max()}
Data source: exoplanets_pscomppars_2026-09-12.csv (GitHub)

FEATURE SETS
------------
A (Orbital): {len(feature_set_a)} features
B (Stellar): {len(feature_set_b)} features
C (Combined): {len(feature_set_c)} features

BEST MODEL
----------
Feature Set: {best_set}
Model: {best_model_name}
Accuracy: {best_config['Accuracy']:.1%}
F1 Score: {best_config['F1']:.3f}

MODEL RANKING
-------------
1. Random Forest: {results_df[results_df['Model']=='Random Forest']['Accuracy'].mean():.1%}
2. Decision Tree: {results_df[results_df['Model']=='Decision Tree']['Accuracy'].mean():.1%}
3. KNN: {results_df[results_df['Model']=='KNN']['Accuracy'].mean():.1%}
4. Baseline: {results_df[results_df['Model']=='Baseline']['Accuracy'].mean():.1%}

FEATURE SET IMPACT
------------------
Combined: {feature_impact['C (Combined)']:.1%}
Orbital: {feature_impact['A (Orbital)']:.1%}
Stellar: {feature_impact['B (Stellar)']:.1%}

KEY FINDINGS
------------
1. Discovery methods are classifiable with {best_config['Accuracy']:.0%} accuracy
2. Tree-based models (RF, DT) outperform linear models
3. Combined features provide best performance
4. Stellar properties alone have limited predictive power
5. Orbital period and temperature are key indicators

LIMITATIONS
-----------
- Top 4 methods only (filtered for class balance)
- Historical data (2011-2026)
- Methods evolve with technology
- Missing values excluded
"""

with open(RESULTS_PATH / 'analysis_summary.txt', 'w') as f:
    f.write(summary)

print(summary)

In [ ]:
print('\nRESULTS GENERATED')
print('='*60)
for file in sorted(RESULTS_PATH.glob('*')):
    size = file.stat().st_size
    print(f'  {file.name} ({size/1024:.1f} KB)')

print('\n[PASS] Notebook execution complete!')